In [17]:
import numpy as np
import pandas as pd
import os
from PIL import Image

from sklearn.model_selection import train_test_split

import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

IMG_SIZE = 224
BATCH_SIZE = 32
DATA_PATH = r"datasets"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device: ", DEVICE)

Device:  cpu


In [18]:
df = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
print("Dimensions: ", df.shape)
df.head()

Dimensions:  (3662, 2)


,id_code,diagnosis
0,000c1434d8d7,2
1,001639a390f0,4
2,0024cdab0c1e,1
3,002c21358ce6,0
4,005b95c28852,0


In [19]:
print("Shape: ", df.shape)
print(df['diagnosis'].value_counts().sort_index())

Shape:  (3662, 2)
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64


In [20]:
train_df, temp_df = train_test_split(df, test_size = 0.3, stratify = df['diagnosis'], random_state = 35)
val_df, test_df = train_test_split(temp_df, test_size = 0.5, stratify = temp_df['diagnosis'], random_state = 35)

In [21]:
class retinalDataset(Dataset):
    def __init__(self, df, images_dir, transform = None):
        self.df = df
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.images_dir, f"{row['id_code']}.png")

        image = Image.open(path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = int(row['diagnosis'])
        return image, label

In [24]:
weights = ResNet18_Weights.IMAGENET1K_V1
transform = weights.transforms()

train_dataset = retinalDataset(train_df, os.path.join(DATA_PATH, 'train_images'), transform)
val_dataset = retinalDataset(val_df, os.path.join(DATA_PATH, 'train_images'), transform)
test_dataset = retinalDataset(test_df, os.path.join(DATA_PATH, 'train_images'), transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [25]:
model = resnet18(weights = weights)
model = model.to(DEVICE)

for parameter in model.parameters():
    parameter.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 5)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Kushagra/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:12<00:00, 3.75MB/s]


In [ ]:
from sklearn.metrics import cohen_kappa_score
from sklearn.utils import compute_class_weight

cw = compute_class_weight('balanced', y = train_dataset['diagnosis'].values, classes = np.arange(5))
class_weights = torch.tensor(cw).to(DEVICE)
criterion = nn.CrossEntropyLoss(weights = class_weights)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params, lr = 0.001)    

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [ ]:
def run_epoch(loader, model, criterion, optimizer = None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss, n = 0, 0
    all_preds, all_labels = [], []
    
    